# 🛒 Milestone 5 — LLMs & RAG
This notebook implements a Retrieval-Augmented Generation (RAG) assistant. We index review texts using a vector search engine (FAISS) based on sentence embeddings, retrieve relevant contexts for user queries, and prompt a local Large Language Model (`Flan-T5`) to generate grounded, fact-accurate answers.

In [2]:
import sys
import os
# Add project root to path so we can import src modules
sys.path.append(os.path.abspath(os.path.join('..')))

import os
import gc
import numpy as np
import pandas as pd

import torch
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Set seed
np.random.seed(42)

## 1. Load Data
Load our processed reviews split.

In [3]:
reviews_path = "data/processed/reviews_split.parquet"
if not os.path.exists(reviews_path):
    reviews_path = "../data/processed/reviews_split.parquet"
if not os.path.exists(reviews_path):
    raise FileNotFoundError("Please run M0 data download first.")

df = pd.read_parquet(reviews_path)
df['text_clean'] = df['text_rev'].fillna("").str.strip()

# Sample reviews to form our RAG database
df_sample = df.sample(n=min(200, len(df)), random_state=42).reset_index(drop=True)
print(f"Database sample size: {len(df_sample)} reviews.")

Database sample size: 200 reviews.


## 2. Build FAISS Vector Search Index

In [4]:
print("Loading Sentence Transformer model locally...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for review database...")
review_texts = df_sample['text_clean'].tolist()
embeddings = embedding_model.encode(review_texts, show_progress_bar=True)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension) # Cosine similarity (Inner Product on normalized vectors)
faiss.normalize_L2(embeddings)
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}.")

Loading Sentence Transformer model locally...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings for review database...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

FAISS index built with 200 vectors of dimension 384.


## 3. Retrieve Context logic

In [5]:
def retrieve_context(query, top_k=3):
    query_vector = embedding_model.encode([query])
    faiss.normalize_L2(query_vector)
    
    scores, indices = index.search(query_vector, top_k)
    contexts = []
    for idx in indices[0]:
        contexts.append(df_sample.iloc[idx]['text_clean'])
    return contexts

# Test retrieval
test_query = "is the bluetooth connection stable on headphones?"
retrieved = retrieve_context(test_query, top_k=2)
print(f"Query: '{test_query}'\n")
for i, ctx in enumerate(retrieved):
    print(f"Result {i+1}: {ctx[:150]}...")

Query: 'is the bluetooth connection stable on headphones?'

Result 1: While they haven't fallen apart yet, they sure feel like they will soon.  The cords are extra thin, the earbud pieces themselves feel rather cheap, li...
Result 2: My son loves these headphones! Great quality and quick shipping! Worth every penny!...


## 4. Setup Local LLM Inference (`Flan-T5-small`)
To save memory and protect local VRAM limits, we load the lightweight `google/flan-t5-small` model.

In [6]:
model_name = "google/flan-t5-small"
print(f"Loading local LLM '{model_name}'...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model.to(device)
print(f"LLM loaded on device: {device}")

Loading local LLM 'google/flan-t5-small'...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

d:\Games\final_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\au\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded on device: cpu


## 5. RAG Generation Pipeline & Structured Summarizer

In [7]:
def answer_query_rag(query):
    # 1. Retrieve matching contexts
    contexts = retrieve_context(query, top_k=3)
    context_block = "\n- ".join(contexts)
    
    # 2. Formulate instruction prompt
    prompt = (
        f"Answer the question based only on the reviews below.\n\n"
        f"Reviews:\n- {context_block}\n\n"
        f"Question: {query}\n"
        f"Answer:"
    )
    
    # 3. Tokenize and generate response
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = llm_model.generate(**inputs, max_new_tokens=100, temperature=0.3)
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Run test queries
print("RAG Query Response:", answer_query_rag("Is this charger fast and worth buying?"))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RAG Query Response: The Lexar jump drive was the fastest of my 3.0 flash drives. And certainly anyone with a 2.0 will want to upgrade.


In [8]:
def summarize_reviews_pros_cons(product_title, reviews_list):
    review_block = "\n- ".join(reviews_list[:5])
    prompt = (
        f"Product: {product_title}\n"
        f"Reviews:\n- {review_block}\n\n"
        f"Summarize the reviews into key Pros and Cons.\n"
        f"Summary:"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = llm_model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test summarizer
sample_reviews = [
    "This adapter charges my iPhone extremely fast, but the cable is too short.",
    "Extremely solid build. Fast charging speed is awesome.",
    "It broke after 2 weeks. Very cheap plastic materials."
]
print("Reviews Summary:\n", summarize_reviews_pros_cons("Fast Charger Adaptor", sample_reviews))

Reviews Summary:
 Great adapter for iPhone.


## 6. Memory Cleanup
Free active memory layers from GPU/VRAM.

In [9]:
# Clear memory contexts
del llm_model
del embedding_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Local models cleared from active memory.")

Local models cleared from active memory.
